***MERGING YEARLY INDICATORS WITH MONTHLY EXPORT DATA***

In [1]:
import pandas as pd

In [3]:
# Load GDP data
gdp_raw = pd.read_excel('/content/gdp_growth.xlsx')
gdp_raw.columns = gdp_raw.columns.str.strip()
gdp_raw.rename(columns={'gdp growth': 'gdp_growth'}, inplace=True)

In [4]:
# Expand quarterly to monthly
quarter_month_map = {'Q1':[1,2,3], 'Q2':[4,5,6], 'Q3':[7,8,9], 'Q4':[10,11,12]}

gdp_monthly_rows = []
for _, row in gdp_raw.iterrows():
    for month in quarter_month_map[row['Quarter']]:
        gdp_monthly_rows.append({
            'Year': int(row['Year']),
            'Month': month,
            'gdp_growth': float(row['gdp_growth'])
        })

gdp_monthly = pd.DataFrame(gdp_monthly_rows)

In [5]:

gdp_monthly = gdp_monthly.sort_values(['Year','Month']).reset_index(drop=True)
gdp_monthly['Date'] = pd.to_datetime(
    gdp_monthly['Year'].astype(str) + '-' + gdp_monthly['Month'].astype(str)
)

gdp_monthly = gdp_monthly.set_index('Date')


gdp_monthly['gdp_growth'] = (
    gdp_monthly['gdp_growth']
    .where(gdp_monthly['gdp_growth'] != gdp_monthly['gdp_growth'].shift(1))
    .interpolate(method='linear')
)


In [6]:
gdp_monthly = gdp_monthly.reset_index()
gdp_monthly = gdp_monthly[['Year','Month','gdp_growth']]

print(gdp_monthly.head(12))

    Year  Month  gdp_growth
0   2018      1    7.700000
1   2018      2    7.766667
2   2018      3    7.833333
3   2018      4    7.900000
4   2018      5    7.633333
5   2018      6    7.366667
6   2018      7    7.100000
7   2018      8    6.600000
8   2018      9    6.100000
9   2018     10    5.600000
10  2018     11    5.466667
11  2018     12    5.333333


In [7]:
# Inflation: monthly directly
inf_raw = pd.read_excel('/content/Inflation.xlsx')
inf_raw.columns = inf_raw.columns.str.strip()
inf_raw.rename(columns={'Inflation': 'Inflation_rate'}, inplace=True)
inf_raw['Inflation_rate'] = pd.to_numeric(inf_raw['Inflation_rate'], errors='coerce')
inf_raw['Inflation_rate'] = inf_raw['Inflation_rate'].ffill()
month_str_map = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,
                 'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}
inf_raw['Month'] = inf_raw['Month'].map(month_str_map)
inf_raw = inf_raw[['Year','Month','Inflation_rate']]
print(f"Inflation: {len(inf_raw)} monthly rows")


Inflation: 96 monthly rows


In [8]:
# REER: monthly directly
reer_raw = pd.read_excel('/content/REER.xlsx')
reer_raw.columns = reer_raw.columns.str.strip()
reer_raw['date']  = pd.to_datetime(reer_raw['date'])
reer_raw['Year']  = reer_raw['date'].dt.year
reer_raw['Month'] = reer_raw['date'].dt.month
reer_raw = reer_raw[['Year','Month','REER']]
print(f"REER: {len(reer_raw)} monthly rows")

REER: 96 monthly rows


In [9]:
# Load monthly export data
export_file    = 'india_monthly_exports_analysis.xlsx'
monthly_exports = pd.read_excel(export_file, sheet_name='Long Format')
if 'Year' not in monthly_exports.columns:
    monthly_exports['Year'] = pd.to_datetime(monthly_exports['Date']).dt.year
if 'Month' not in monthly_exports.columns:
    monthly_exports['Month'] = pd.to_datetime(monthly_exports['Date']).dt.month


In [10]:
# Merging
merged_data = monthly_exports.merge(gdp_monthly, on=['Year','Month'], how='left')
merged_data = merged_data.merge(inf_raw,         on=['Year','Month'], how='left')
merged_data = merged_data.merge(reer_raw,        on=['Year','Month'], how='left')

print(f"\nMerged Data Shape: {merged_data.shape}")
print(f"Columns: {list(merged_data.columns)}")
missing_indicators = merged_data[['gdp_growth','Inflation_rate','REER']].isnull().sum()
print(f"\nMissing values after merge:\n{missing_indicators}")



Merged Data Shape: (24096, 13)
Columns: ['Country', 'Export_Value', 'Year', 'Month', 'Month_Name', 'Date', 'Year_Month', 'Monthly_Total', 'Export_Share_%', 'MoM Growth (%)', 'gdp_growth', 'Inflation_rate', 'REER']

Missing values after merge:
gdp_growth        0
Inflation_rate    0
REER              0
dtype: int64


In [11]:
output_file = 'india_monthly_exports_with_indicators.xlsx'
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    merged_data.to_excel(writer, sheet_name='Complete Data', index=False)
    monthly_totals = merged_data.groupby('Date').agg(
        Export_Value  =('Export_Value',   'sum'),
        gdp_growth    =('gdp_growth',     'first'),
        Inflation_rate=('Inflation_rate', 'first'),
        REER          =('REER',           'first')
    ).reset_index()
    monthly_totals.to_excel(writer, sheet_name='Monthly Totals', index=False)

***CORRELATION ANALYSIS: EXPORTS vs MACRO INDICATORS***

In [13]:
import scipy.stats as stats

input_file = 'india_monthly_exports_with_indicators.xlsx'
df = pd.read_excel(input_file, sheet_name='Complete Data')
df.columns = df.columns.str.strip()
df['Date'] = pd.to_datetime(df['Date'])

# Build monthly
monthly = (df
    .groupby('Date')
    .agg(
        Total_Export   = ('Export_Value',   'sum'),
        GDP_Growth     = ('gdp_growth',     'first'),
        Inflation_Rate = ('Inflation_rate', 'first'),
        REER           = ('REER',           'first')
    )
    .reset_index()
    .sort_values('Date')
)

print(f"Monthly correlation dataset : {len(monthly)} observations")
print(f"Period : {monthly['Date'].min().strftime('%b %Y')} "
      f"→ {monthly['Date'].max().strftime('%b %Y')}")
print(f"\nDescriptive Statistics:")
print(monthly[['Total_Export','GDP_Growth','Inflation_Rate','REER']].describe().round(2))

#  Pearson correlation &ignificance
print(f"  PEARSON CORRELATION  —  MONTHLY  (n = {len(monthly)})")
print(f"  {'Pair':<40} {'r':>8} {'p-value':>10} {'Sig':>6}")


pairs = [
    ('Total_Export',   'GDP_Growth',     'Export ~ GDP Growth'),
    ('Total_Export',   'Inflation_Rate', 'Export ~ Inflation'),
    ('Total_Export',   'REER',           'Export ~ REER'),
    ('GDP_Growth',     'Inflation_Rate', 'GDP Growth ~ Inflation'),
    ('GDP_Growth',     'REER',           'GDP Growth ~ REER'),
    ('Inflation_Rate', 'REER',           'Inflation ~ REER'),
]

corr_results = []
for x_col, y_col, label in pairs:
    r, p = stats.pearsonr(monthly[x_col].values, monthly[y_col].values)
    sig  = ('***' if p < 0.001 else '**' if p < 0.01
            else '*' if p < 0.05 else 'n.s.')
    corr_results.append({'Pair': label, 'r': round(r,4),
                         'p_value': round(p,4), 'Sig': sig})
    print(f"  {label:<40} {r:>8.4f} {p:>10.4f} {sig:>6}")

print(f"\n  Significance: *** p<0.001  ** p<0.01  * p<0.05  n.s. not significant")
print(f"  Critical |r| for significance at α=0.05 (n={len(monthly)}, two-tailed): ±0.201")

# Full correlation matrix
rename_map = {
    'Total_Export':   'Total Export',
    'GDP_Growth':     'GDP Growth',
    'Inflation_Rate': 'Inflation',
    'REER':           'REER'
}
corr_matrix = (monthly[list(rename_map.keys())]
               .rename(columns=rename_map)
               .corr()
               .round(4))

print(f"\nFull Correlation Matrix (Monthly, n={len(monthly)}):")
print(corr_matrix.to_string())

Monthly correlation dataset : 96 observations
Period : Jan 2018 → Dec 2025

Descriptive Statistics:
       Total_Export  GDP_Growth  Inflation_Rate    REER
count         96.00       96.00           96.00   96.00
mean       32150.60        5.75            4.85   99.51
std         6228.88        5.57            1.70    2.43
min        10173.28      -23.90            0.25   92.89
25%        26818.12        5.18            3.68   97.82
50%        33589.71        7.00            4.97   99.82
75%        37017.59        7.67            6.10  101.29
max        44573.67       20.10            7.79  104.72
  PEARSON CORRELATION  —  MONTHLY  (n = 96)
  Pair                                            r    p-value    Sig
  Export ~ GDP Growth                        0.5591     0.0000    ***
  Export ~ Inflation                         0.0212     0.8373   n.s.
  Export ~ REER                              0.1074     0.2976   n.s.
  GDP Growth ~ Inflation                    -0.1667     0.1046   n.s.
  